In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch00-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "00-03-research-pain-points.py"

# 化學與材料研發的現場痛點

先從上到下執行。下格將分析程式寫入檔案，最後一格才使用安裝好的環境執行它。

資料為 DeepChem 載入的 ESOL，保留全部紀錄，不訓練模型、不生成假量測值。原始研究：[Delaney 2004](https://doi.org/10.1021/ci034243x)；資料彙整：[MoleculeNet](https://doi.org/10.1039/C7SC02664A)。

Colab 使用 CPU 即可。若看不到 2026.07，請先核對書籍環境說明，勿忽略版本檢查。

<!-- concept-ch00-05:start -->
資料筆數以外，還要檢查每筆紀錄增加了哪種經驗。

| 看到的紀錄 | 仍缺少的資訊 | 直接可做的整理 |
|---|---|---|
| 同系列類似物很多 | 其他骨架的表現 | 按骨架計數並列出空白範圍 |
| 只留下成功反應 | 失敗時用了哪些條件 | 保留失敗原因與原始條件 |
| 名稱和結果散在不同檔案 | 結果屬於哪個樣品 | 用樣品編號連回來源 |

例如第一列的問題，增加相同系列的紀錄可以讓該系列更完整，卻沒有替另一類骨架增加觀測。


<!-- concept-ch00-05:end -->

`table["parse_ok"]` 記錄每筆 SMILES 是否成功讀入；對這欄呼叫 `.sum()`，會把 `True` 當作 1、`False` 當作 0，因此得到成功讀入的筆數。`np.isfinite(table["logS"])` 檢查每個量測值是否為有限數值，會排除缺值 `NaN` 和正負無限大。`table["weight"] > 0` 則檢查是否有可用的實測值；這裡的權重不是分子的重量。最後，`table["canonical_smiles"].nunique()` 計算不同標準 SMILES 的個數，預設不計缺值。


In [ ]:
%%writefile "{script}"
from pathlib import Path
import json
import numpy as np
import pandas as pd
import os
# 使用適合輸出圖檔的繪圖後端，避免沿用 Colab 的顯示設定。
os.environ["MPLBACKEND"] = "Agg"
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem import Descriptors, Draw
import deepchem as dc

# ESOL 真實水溶解度；原始研究 DOI:10.1021/ci034243x。
# 本章只讀取、核對與篩選，不訓練模型，因此不切分資料；不轉換標籤。
tasks, parts, transformers = dc.molnet.load_delaney(
    featurizer=dc.feat.RawFeaturizer(smiles=True), splitter=None,
    transformers=[], reload=False)
data = parts[0]
assert len(tasks) == 1 and not transformers
mols = [Chem.MolFromSmiles(str(s)) for s in data.ids]
table = pd.DataFrame({"row_id": np.arange(len(data)), "smiles": data.ids,
    "logS": data.y[:, 0], "weight": data.w[:, 0],
    "parse_ok": [m is not None for m in mols]})
table["canonical_smiles"] = [Chem.MolToSmiles(m) if m is not None else None for m in mols]
table["MW"] = [Descriptors.MolWt(m) if m is not None else np.nan for m in mols]
book = Path("../..").resolve() if Path("../../_quarto.yml").is_file() else Path.cwd()
figures = book / "figures"
figures.mkdir(exist_ok=True)
outputs = Path("outputs")
outputs.mkdir(exist_ok=True)
print("Python / RDKit:", __import__("sys").version.split()[0], __import__("rdkit").__version__)
print("Task:", tasks[0], "rows:", len(table))
counts = {"Records": len(table), "Parsed": int(table["parse_ok"].sum()),
    "Finite labels": int(np.isfinite(table["logS"]).sum()),
    "Positive weight": int((table["weight"] > 0).sum()),
    "Unique structures": int(table["canonical_smiles"].nunique())}
print(json.dumps(counts, ensure_ascii=False))
duplicates = table.loc[table["canonical_smiles"].duplicated(keep=False)].sort_values("canonical_smiles")
print("Repeated-structure rows:", len(duplicates))
print(duplicates[["row_id","canonical_smiles","logS"]].head(8).to_string(index=False))
fig, ax = plt.subplots(figsize=(7,4.5))
ax.barh(list(counts), list(counts.values()), color="#287d8e")
for i,v in enumerate(counts.values()): ax.text(v+8,i,str(v),va="center")
ax.set_xlim(0,len(table)*1.17);ax.set_xlabel("Count (records or distinct structures)")
fig.tight_layout();fig.savefig(figures / "00-03-audit.png",dpi=170);plt.close(fig)


In [ ]:
from IPython.display import Image, display
run = subprocess.run([python, str(script)], capture_output=True, text=True)
if run.returncode:
    print(run.stderr)
    run.check_returncode()
print(run.stdout.replace("This module requires PyTorch to be installed.\n", "").strip())
figure_dir = Path("../../figures") if Path("../../_quarto.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "00-03-audit.png")))